In [10]:
import sys
import os

# Ativa o recarregamento automático
%load_ext autoreload
%autoreload 2

# Define os caminhos absolutos necessários
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), "..", "..")) # D:\GeoPipe
pasta_src = os.path.join(raiz_projeto, "src")                         # D:\GeoPipe\src

# Adiciona ambos ao sys.path se já não estiverem lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)
if pasta_src not in sys.path:
    sys.path.append(pasta_src)

# Agora o import vai funcionar e o 'nodes.py' vai encontrar a pasta 'utils'
from src.fmask_pipeline.pipelines.cloud_preprocess.nodes import cloud_removal


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import pandas as pd
import os

## Copy images

In [57]:
temp_image_dir = "../data/temp/interpolation/images/all"
temp_mask_dir = "../data/temp/interpolation/masks/all"
temp_clean_image_dir = "../data/temp/interpolation/clean_images/all"
temp_color_log_dir = "../data/temp/interpolation/color_logs/all"

In [58]:
os.makedirs(temp_image_dir, exist_ok=True)
os.makedirs(temp_mask_dir, exist_ok=True)
os.makedirs(temp_clean_image_dir, exist_ok=True)
os.makedirs(temp_color_log_dir, exist_ok=True)

In [59]:
images = pd.read_csv("../data/temporal_interpolation_images.csv")

In [60]:
images.columns

Index(['mask', 'cloud_percentage', 'cloud_shadow_percentage', 'reservoir',
       'image_path', 'random_mask', 'random_cloud_percentage',
       'random_cloud_shadow_percentage'],
      dtype='object')

In [61]:
images.iloc[0, 4], images.iloc[0, 5]

('D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190607.tif',
 'D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\\2025\\mask_sentinel_TOA_S2_argemiro_20251209.tif')

In [62]:
images_path = images["image_path"].tolist()
masks_path = images["random_mask"].tolist()

In [63]:
images_path[0], masks_path[0] 

('D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190607.tif',
 'D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\\2025\\mask_sentinel_TOA_S2_argemiro_20251209.tif')

In [71]:
all_dates = [image.split('_')[-1].split('.')[0] for image in images_path]
all_dates = sorted(list(set(all_dates)))

init_date = all_dates[0]
final_date = all_dates[-1]

# format data to yyyy-mm-dd
init_date_formatted = f"{init_date[:4]}-{init_date[4:6]}-{init_date[6:]}"
final_date_formatted = f"{final_date[:4]}-{final_date[4:6]}-{final_date[6:]}"

print(f"Initial date: {init_date_formatted}")
print(f"Final date: {final_date_formatted}")

Initial date: 2017-05-16
Final date: 2026-06-21


In [76]:
# year folders
for year in range(int(init_date[:4]), int(final_date[:4]) + 1):
    os.makedirs(os.path.join(temp_image_dir, str(year), 'fmask'), exist_ok=True)
    os.makedirs(os.path.join(temp_mask_dir, str(year), 'fmask'), exist_ok=True)
    os.makedirs(os.path.join(temp_clean_image_dir, str(year), 'fmask'), exist_ok=True)
    os.makedirs(os.path.join(temp_color_log_dir, str(year), 'fmask'), exist_ok=True)

In [ ]:
import os
import shutil

for image_path, mask_path in zip(images_path[:5], masks_path[:5]):
    image_year = image_path.split('_')[-1].split('.')[0][:4]
    mask_year = mask_path.split('_')[-1].split('.')[0][:4]
    
    image_filename = os.path.basename(image_path)
    mask_filename = os.path.basename(mask_path)

    temp_image_path = os.path.join(temp_image_dir, image_year, image_filename)
    temp_mask_path = os.path.join(temp_mask_dir, mask_year, mask_filename)

    # Garante que as pastas de destino existam antes de copiar
    os.makedirs(temp_image_dir, exist_ok=True)
    os.makedirs(temp_mask_dir, exist_ok=True)

    if not os.path.exists(temp_image_path):
        shutil.copy2(image_path, temp_image_path)  # Copia nativa do Python (funciona em qualquer OS)
        
    if not os.path.exists(temp_mask_path):
        shutil.copy2(mask_path, temp_mask_path)


KeyboardInterrupt: 

## Clean images

In [ ]:
cloud_removal(
    path_images = temp_image_dir.replace("all", ""),
    path_masks = temp_mask_dir.replace("all", ""),
    output_path = temp_clean_image_dir.replace("all", ""),
    location_name = "all",
    cloud_and_cloud_shadow_pixels = [1, 2],
    init_date = init_date_formatted,
    final_date = final_date_formatted,
    skip_clean = False,
    color_file_log_path = temp_color_log_dir.replace("all", ""),
    cloud_mask_algorithm = "fmask",
    reconstruction_algorithm = "temporal_interpolation",
    max_workers = 4,
)

Cleaning Images:   0%|          | 1/1578 [00:01<29:36,  1.13s/file]Failed to process 'sentinel_BOA_S2_SR_engenheiro_avidos_20170812.tif': [WinError 3] O sistema não pode encontrar o caminho especificado: '../data/temp/interpolation/masks/all\\fmask\\2017\\'
Failed to process 'sentinel_BOA_S2_SR_engenheiro_avidos_20170906.tif': [WinError 3] O sistema não pode encontrar o caminho especificado: '../data/temp/interpolation/masks/all\\fmask\\2017\\'
Failed to process 'sentinel_BOA_S2_SR_engenheiro_avidos_20170926.tif': [WinError 3] O sistema não pode encontrar o caminho especificado: '../data/temp/interpolation/masks/all\\fmask\\2017\\'
Failed to process 'sentinel_BOA_S2_SR_engenheiro_avidos_20171105.tif': [WinError 3] O sistema não pode encontrar o caminho especificado: '../data/temp/interpolation/masks/all\\fmask\\2017\\'
Failed to process 'sentinel_BOA_S2_SR_lagoa_do_arroz_20170817.tif': [WinError 3] O sistema não pode encontrar o caminho especificado: '../data/temp/interpolation/masks/a

True